# 06_embeddings_store

对应 `index.md` 第 6 阶段：Embedding、向量库写入 / 查询。

笔记本在 `codes/langchain/06_embeddings_store/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

DeepSeek 目前没有 Embedding API。本阶段用 `DeterministicFakeEmbedding`（固定假向量，**不是** FakeModel）练 `Embeddings` 与 `InMemoryVectorStore` 接口；换真 Embedding 模型时调用方式相同。下一阶段 RAG 再接真实 ChatModel。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "06_embeddings_store" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "06_embeddings_store"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/06_embeddings_store')

## 1. embed_query / embed_documents

用 `DeterministicFakeEmbedding(size=16)` 建 `emb`。

- 对字符串 `"hello"` 调用 `embed_query`，打印向量长度。
- 对 `['alpha', 'beta']` 调用 `embed_documents`，打印返回列表长度，以及第一个向量的长度。


In [5]:
from langchain_core.embeddings import DeterministicFakeEmbedding

# 作答

emb = DeterministicFakeEmbedding(size=16)

query_emb = emb.embed_query("hello")

print(len(query_emb))
print(query_emb)

docs_emb = emb.embed_documents(["alpha", "beta"])

print(len(docs_emb))
print(len(docs_emb[0]))
#评阅
# 对。size=16，embed_query / embed_documents 长度都打对了。多打印整段向量无妨。

#参考答案
# emb = DeterministicFakeEmbedding(size=16)
# print(len(emb.embed_query("hello")))
# docs_emb = emb.embed_documents(["alpha", "beta"])
# print(len(docs_emb), len(docs_emb[0]))


16
[np.float64(0.08416199374898006), np.float64(0.9044802814015355), np.float64(0.9572251281704619), np.float64(-1.6774928811602647), np.float64(-0.42073552411521187), np.float64(-2.1036701523314725), np.float64(-0.47866384813518625), np.float64(-0.7427188130087025), np.float64(1.6837451654921356), np.float64(1.8576430609830885), np.float64(2.12603870959233), np.float64(-0.630965161266994), np.float64(-0.9385258201571044), np.float64(-0.30585771466884054), np.float64(-0.3832711254134778), np.float64(-2.472736611178281)]
2
16


## 2. from_documents 写入

已有 `docs` 与 `emb`。用 `InMemoryVectorStore.from_documents(docs, emb)` 得到 `store`。

用 `similarity_search("猫", k=1)` 查一次，打印命中文档的 `page_content` 与 `metadata`。


In [ ]:
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding, Embeddings
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]

# 作答

vs = InMemoryVectorStore.from_documents(docs, emb)

retrived = vs.similarity_search("猫", k = 1)

for doc in retrived:
    print(doc.page_content)
    print(doc.metadata)

class CustomEmbedding(Embeddings):
    @staticmethod
    def embed_query(text: str) -> list[float]:
        if "猫" in text:
            return [0.1, 0.2, 0.3]
        else:
            return [0.4, 0.5, 0.6]

    @staticmethod
    def embed_documents(texts: list[str]) -> list[list[float]]:
        return [CustomEmbedding.embed_query(text) for text in texts]

vs = InMemoryVectorStore.from_documents(docs, CustomEmbedding())

retrived = vs.similarity_search("猫", k = 1)

for doc in retrived:
    print(doc.page_content)
    print(doc.metadata)
#评阅
# 对。from_documents + similarity_search 满足题意；假向量命中「狗」正常。后面的 CustomEmbedding 是加练，无妨。

#参考答案
# store = InMemoryVectorStore.from_documents(docs, emb)
# hit = store.similarity_search("猫", k=1)[0]
# print(hit.page_content)
# print(hit.metadata)

狗喜欢追球。
{'source': 'b.txt'}
猫喜欢吃鱼。
{'source': 'a.txt'}


## 3. similarity_search_with_score

已有写入好的 `store`。对查询 `"向量检索"` 调用 `similarity_search_with_score(..., k=2)`。

对每个 `(doc, score)`，打印 `doc.page_content` 与 `score`。


In [15]:
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
docs = [
    Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
    Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
]
store = InMemoryVectorStore.from_documents(docs, emb)

# 作答

results = store.similarity_search_with_score("向量检索", k = 2)

for doc, score in results:
    print(doc.page_content)
    print(score)
#评阅
# 对。with_score、k=2，page_content 与 score 都打了。

#参考答案
# for doc, score in store.similarity_search_with_score("向量检索", k=2):
#     print(doc.page_content, score)


狗喜欢追球。
0.19105279104200798
向量库用来做相似度检索。
0.1068992333730228


## 4. add_documents 追加

已有 `store`。再追加一个 `Document`：`page_content` 为 `新写入的 embedding 说明。`，`metadata` 为 `{"source": "d.txt"}`。

打印 `add_documents` 返回的 id 列表；再用 `similarity_search("embedding", k=1)` 打印命中的 `page_content`。


In [17]:
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
store = InMemoryVectorStore.from_documents(
    [
        Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
        Document(page_content="狗喜欢追球。", metadata={"source": "b.txt"}),
    ],
    emb,
)

# 作答

ids = store.add_documents([
    Document(page_content="新写入的 embedding 说明。", metadata={"source": "d.txt"}),
])
print(ids)

retrived = store.similarity_search("embedding", k = 1)

for doc in retrived:
    print(doc.page_content)
#评阅
# 对。追加、打印 id、再检索都齐。

#参考答案
# ids = store.add_documents([
#     Document(page_content="新写入的 embedding 说明。", metadata={"source": "d.txt"}),
# ])
# print(ids)
# print(store.similarity_search("embedding", k=1)[0].page_content)


['c90d6657-2604-4087-94dd-420ff516a89e']
新写入的 embedding 说明。


## 5. add_texts

用 `InMemoryVectorStore(embedding=emb)` 建空库，再调用 `add_texts`：

- 文本：`["苹果派", "向量检索"]`
- `metadatas`：`[{"tag": "food"}, {"tag": "rag"}]`

打印返回的 id 列表；再用 `similarity_search("检索", k=1)` 打印命中文档的 `page_content` 与 `metadata`。


In [ ]:
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)

# 作答

vs = InMemoryVectorStore(embedding=emb)

ids = vs.add_texts(
    ["苹果派", "向量检索"],
    [{"tag": "food"}, {"tag": "rag"}]
)

retrived = vs.similarity_search("检索", k = 1)

for doc in retrived:
    print(doc.page_content)
    print(doc.metadata)
#评阅
# 缺一半。add_texts 与检索对了，但题面还要打印返回的 id 列表（有 ids 变量却没 print）。

#参考答案
# store = InMemoryVectorStore(embedding=emb)
# ids = store.add_texts(
#     ["苹果派", "向量检索"],
#     metadatas=[{"tag": "food"}, {"tag": "rag"}],
# )
# print(ids)
# hit = store.similarity_search("检索", k=1)[0]
# print(hit.page_content, hit.metadata)


## 6. as_retriever

已有 `store`。用 `as_retriever(search_kwargs={"k": 1})` 得到 `retriever`，对字符串 `"向量"` 调用 `invoke`。

打印返回列表长度，以及第一项的 `page_content`。


In [18]:
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

emb = DeterministicFakeEmbedding(size=32)
store = InMemoryVectorStore.from_documents(
    [
        Document(page_content="猫喜欢吃鱼。", metadata={"source": "a.txt"}),
        Document(page_content="向量库用来做相似度检索。", metadata={"source": "c.txt"}),
    ],
    emb,
)

# 作答
retrived = store.as_retriever(search_kwargs={"k": 1}).invoke("向量")

print(len(retrived))
print(retrived[0].page_content)
#评阅
# 对。as_retriever(k=1) + invoke，长度与 page_content 都打了。

#参考答案
# retriever = store.as_retriever(search_kwargs={"k": 1})
# docs = retriever.invoke("向量")
# print(len(docs))
# print(docs[0].page_content)


1
向量库用来做相似度检索。
